# 01 XR WRIST exploratory data analysis

This notebook uses only the generated manifest. It saves concise figures and displays several NORMAL and ABNORMAL image examples without making clinical claims.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
manifest_path = ROOT / "data/manifests/wrist_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("Run 00_data_access_audit.ipynb first.")
manifest = pd.read_csv(manifest_path)
figures = ROOT / "reports/figures"
figures.mkdir(parents=True, exist_ok=True)
manifest.head()


In [ ]:
# Study-level class balance and views per study
studies = manifest.drop_duplicates("study_id").copy()
summary = {
    "patients": int(studies["patient_id"].nunique()),
    "studies": int(studies["study_id"].nunique()),
    "images": int(len(manifest)),
    "normal_studies": int((studies["label"] == 0).sum()),
    "abnormal_studies": int((studies["label"] == 1).sum()),
}
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
studies["label"].map({0: "NORMAL", 1: "ABNORMAL"}).value_counts().reindex(["NORMAL", "ABNORMAL"]).plot.bar(ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Study-level class balance")
axes[0].set_ylabel("Studies")
studies["num_views_in_study"].plot.hist(ax=axes[1], bins=range(1, int(studies["num_views_in_study"].max()) + 2), color="#72B7B2")
axes[1].set_title("Views per study")
axes[1].set_xlabel("Views")
fig.tight_layout()
fig.savefig(figures / "eda_class_balance_and_views.png", dpi=160)
plt.show()


In [ ]:
# Measured image-size distribution; unreadable files are skipped here because the audit reports them.
sizes = []
for image_path in manifest["image_path"]:
    try:
        with Image.open(image_path) as image:
            sizes.append(image.size)
    except (OSError, ValueError):
        pass
sizes_df = pd.DataFrame(sizes, columns=["width", "height"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sizes_df["width"].plot.hist(ax=axes[0], bins=30, color="#4C78A8")
sizes_df["height"].plot.hist(ax=axes[1], bins=30, color="#F58518")
axes[0].set_title("Image width distribution")
axes[1].set_title("Image height distribution")
fig.tight_layout()
fig.savefig(figures / "eda_image_sizes.png", dpi=160)
plt.show()
sizes_df.describe()


In [ ]:
# Several representative input examples sampled by label. These are dataset examples, not localized explanations.
sample_rows = (
    manifest.groupby("label", group_keys=False)
    .apply(lambda frame: frame.sample(n=min(4, len(frame)), random_state=23))
    .reset_index(drop=True)
)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.ravel(), sample_rows.iterrows()):
    with Image.open(row["image_path"]) as image:
        ax.imshow(image.convert("L"), cmap="gray")
    ax.set_title(f'{"ABNORMAL" if row["label"] else "NORMAL"}\n{row["study_id"]}')
    ax.axis("off")
for ax in axes.ravel()[len(sample_rows):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(figures / "eda_examples.png", dpi=160)
plt.show()
